# Fine-tune Cross-Encoder CV-JD v0.5

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **MSE regression loss** trên dataset v0.4 đã relabel bằng LLM.

| | |
|---|---|
| **Dataset** | v0.4 — 4900 train / 1050 validation / 1050 test (7000 pairs, 4188 relabeled by LLM) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` regression (label = score / 100) |
| **Evaluator** | Spearman correlation + LabelAcc |
| **max_length** | 512 tokens |
| **Branch** | `develop` |

**Mục tiêu**: vượt LabelAcc 60.76% của v0.2 nhờ data quality cao hơn (relabeled 4188/7000 pairs bởi LLM với rubric v0.2).

In [20]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.5
!git pull origin experiment/cross-encoder-v0.5

/content/Ai-Recruiter-Mini-Ai-Service
Already on 'experiment/cross-encoder-v0.5'
Your branch is up to date with 'origin/experiment/cross-encoder-v0.5'.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 3), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 417 bytes | 417.00 KiB/s, done.
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.5 -> FETCH_HEAD
   fe9e737..37f0022  experiment/cross-encoder-v0.5 -> origin/experiment/cross-encoder-v0.5
Updating fe9e737..37f0022
Fast-forward
 training/fine_tune_cross_encoder.py | 16 +---------------
 1 file changed, 1 insertion(+), 15 deletions(-)


In [21]:
!pip install -r requirements.txt

In [22]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.4/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

train       :  4900 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
validation  :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
test        :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


In [25]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


## Debug run — sanity check (1 epoch, 40 samples)

In [26]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.4/cross_encoder \
    --loss mse \
    --evaluator spearman \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20

2026-06-20 15:12:04.047482: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.4/cross_encoder
Output dir : artifacts/models/cross-encoder-cv-jd-v0.1
Loss       : mse  (MSELoss)
Evaluator  : spearman
Epochs     : 1  |  Batch size: 4  |  Max length: 512

Train: 40 pairs  |  Val: 20 pairs

Steps/epoch: 10  |  Warmup steps: 1

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:234: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  epoch  1/1  step    10  val_spearman=-0.1982
  [done]  val_spearman=-0.1982

Fine-t

## Full training — 10 epochs, save to Google Drive

In [30]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

import os, time
os.makedirs(f"{drive_base}/models/cross-encoder-cv-jd-v0.5", exist_ok=True)
time.sleep(3)

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.4/cross_encoder \
    --loss boundary \
    --evaluator label_acc \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.5 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.5_report.json \
    --epochs 15 \
    --batch-size 16


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-06-20 15:31:19.654891: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.4/cross_encoder
Output dir : /content/drive/MyDrive/ai-recruiter/models/cross-encoder-cv-jd-v0.5
Loss       : boundary  (BoundaryAwareLoss)
Evaluator  : label_acc
Epochs     : 15  |  Batch size: 16  |  Max length: 512

Train: 4900 pairs  |  Val: 1050 pairs

Steps/epoch: 307  |  Warmup steps: 30

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:234: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Pleas

In [28]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.5_report.json").read_text(encoding="utf-8")
)
print(f"Base model : {report['base_model']}")
print(f"Loss       : {report['loss']}")
print()
print(json.dumps(report["metrics"], indent=2))

Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Loss       : boundary

{
  "validation": {
    "mae": 11.6816,
    "rmse": 15.3355,
    "label_accuracy": 0.5857,
    "pair_count": 1050,
    "mean_predicted_score": 50.0001,
    "mean_target_score": 49.919
  },
  "test": {
    "mae": 11.7442,
    "rmse": 16.2042,
    "label_accuracy": 0.5743,
    "pair_count": 1050,
    "mean_predicted_score": 51.4439,
    "mean_target_score": 53.28
  }
}


In [29]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.5_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.5_report.json",
)
print(f"Saved to {reports_dir}")

Saved to /content/drive/MyDrive/ai-recruiter/reports
